# Q3 — Huấn luyện CNN bằng PyTorch

Ở Q1, bạn đã tự cài đặt convolution và max pooling bằng NumPy.

Trong Q3, ta chuyển sang PyTorch để tập trung vào các quyết định ở mức mô hình:

- chọn kiến trúc CNN;
- theo dõi loss và độ chính xác;
- dùng Batch Normalization và Dropout;
- điều chỉnh một số siêu tham số;
- lựa chọn mô hình bằng tập kiểm định;
- chỉ đánh giá tập kiểm tra ở bước cuối.

Dữ liệu sử dụng là **CIFAR-10**, gồm ảnh màu $32\times32$ thuộc 10 lớp.

In [ ]:
from pathlib import Path

assert Path("dl2").exists(), (
    "Không tìm thấy thư mục dl2/. "
    "Hãy chạy notebook từ thư mục gốc của Assignment 2."
)

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from dl2.data import (
    get_cifar10_loaders,
    CIFAR10_CLASSES,
)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

## 1. Ôn lại cấu trúc của một CNN

Một CNN phân loại ảnh thường có hai phần.

### Bộ trích xuất đặc trưng (feature extractor)

Nhiều khối:

$$
\text{Conv}
\rightarrow
\text{Normalization}
\rightarrow
\text{ReLU}
\rightarrow
\text{Pooling}.
$$

Các lớp đầu thường học cạnh và texture. Các lớp sâu hơn kết hợp chúng thành đặc trưng
phức tạp hơn.

### Bộ phân loại (classifier)

Feature maps cuối cùng được đưa về một vector và ánh xạ thành 10 logits.

Trong notebook này, ta dùng **Global Average Pooling** thông qua
`AdaptiveAvgPool2d(1)`. Mỗi feature map được rút thành một số duy nhất, giúp giảm số
tham số của classifier.

## 2. Đếm tham số trước khi huấn luyện

Nếu một lớp convolution có:

- $C_{in}$ kênh đầu vào;
- $C_{out}$ bộ lọc;
- kernel $K\times K$;

thì số trọng số là:

$$
C_{out}C_{in}K^2.
$$

Nếu có bias:

$$
\text{parameters}
=
C_{out}C_{in}K^2+C_{out}.
$$

So với fully connected trên toàn ảnh, chia sẻ trọng số giúp số tham số nhỏ hơn rất nhiều.

### Câu hỏi 1 — Số tham số

Một lớp `Conv2d(3, 32, kernel_size=3, padding=1)` có bao nhiêu tham số nếu có bias?

So sánh với một lớp fully connected ánh xạ trực tiếp ảnh CIFAR-10 đã làm phẳng
($3\times32\times32$) sang 32 đầu ra.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 3. Tải dữ liệu

Ta dùng một cách chia train/validation cố định để các thí nghiệm có thể so sánh được.

Không dùng tập kiểm tra (test set) để lựa chọn siêu tham số.

In [ ]:
train_loader, val_loader, test_loader = (
    get_cifar10_loaders(
        batch_size=128,
        seed=42,
    )
)

x_batch, y_batch = next(
    iter(train_loader)
)

print("Ảnh:", x_batch.shape)
print("Nhãn:", y_batch.shape)

### Quan sát một số ảnh

In [ ]:
# Hoàn nguyên chuẩn hóa chỉ để hiển thị.
mean = torch.tensor(
    [0.4914, 0.4822, 0.4465]
).view(3, 1, 1)

std = torch.tensor(
    [0.2470, 0.2435, 0.2616]
).view(3, 1, 1)

fig = plt.figure(
    figsize=(10, 5)
)

for i in range(10):
    ax = fig.add_subplot(
        2,
        5,
        i + 1,
    )

    img = (
        x_batch[i].cpu()
        * std
        + mean
    ).clamp(0, 1)

    ax.imshow(
        img.permute(
            1,
            2,
            0,
        )
    )
    ax.set_title(
        CIFAR10_CLASSES[
            int(y_batch[i])
        ]
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Thiết kế `SmallCNN`

Bạn sẽ cài đặt một CNN nhỏ với ba mức đặc trưng.

Yêu cầu:

- đầu vào: `(N, 3, 32, 32)`;
- hỗ trợ `use_batchnorm=True/False`;
- hỗ trợ `dropout_p`;
- `forward_features(x)` trả vector đặc trưng cuối có 128 chiều;
- `forward(x)` trả logits `(N, 10)`.

Gợi ý cấu trúc tổng quát:

$$
3
\rightarrow
32
\rightarrow
64
\rightarrow
128
\rightarrow
10.
$$

Bạn tự quyết định vị trí hợp lý cho pooling, BatchNorm và Dropout dựa trên kiến thức đã
học. Không cần cố tạo mạng quá sâu.

### Thực hành 1 — Cài đặt `SmallCNN`

In [ ]:
class SmallCNN(nn.Module):
    def __init__(
        self,
        use_batchnorm=True,
        dropout_p=0.1,
    ):
        super().__init__()

        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # Xây bộ trích xuất đặc trưng.
        #
        # Yêu cầu:
        # - nhận ảnh 3x32x32;
        # - feature vector cuối có 128 chiều;
        # - có thể bật/tắt BatchNorm;
        # - có thể điều chỉnh Dropout.
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    @torch.jit.export
    def forward_features(self, x):
        # ============================================================
        # TODO: CODE CỦA BẠN
        # Trả về feature vector kích thước (N, 128).
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    def forward(self, x):
        # ============================================================
        # TODO: CODE CỦA BẠN
        # Trả logits kích thước (N, 10).
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

### Kiểm tra kích thước

In [ ]:
shape_model = SmallCNN(
    use_batchnorm=True,
    dropout_p=0.1,
)

dummy = torch.randn(
    4,
    3,
    32,
    32,
)

features = shape_model.forward_features(
    dummy
)
logits = shape_model(
    dummy
)

print(
    "Feature shape:",
    features.shape,
)
print(
    "Logit shape:",
    logits.shape,
)

assert features.shape == (
    4,
    128,
)
assert logits.shape == (
    4,
    10,
)

print("✓ Kích thước mô hình hợp lệ")

## 5. Vòng lặp huấn luyện

Các bước về mặt khái niệm vẫn giống Assignment 1:

$$
\text{forward}
\rightarrow
\text{loss}
\rightarrow
\text{backward}
\rightarrow
\text{update}.
$$

Điểm cần chú ý với CNN:

- chuyển ảnh và nhãn lên đúng device;
- gọi `model.train()` khi huấn luyện;
- gọi `model.eval()` khi đánh giá;
- không tính gradient trong validation/test;
- lưu cả loss và accuracy.

### Thực hành 2 — `train_one_epoch`

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        # ============================================================
        # TODO: CODE CỦA BẠN
        #
        # Hoàn thiện một mini-batch huấn luyện và cập nhật thống kê.
        # ============================================================
        raise NotImplementedError
        # ============================================================
        # KẾT THÚC CODE CỦA BẠN
        # ============================================================

    return (
        total_loss / total,
        correct / total,
    )

### Thực hành 3 — `evaluate`

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    # ============================================================
    # TODO: CODE CỦA BẠN
    #
    # Tính loss và accuracy mà không cập nhật tham số.
    # ============================================================
    raise NotImplementedError
    # ============================================================
    # KẾT THÚC CODE CỦA BẠN
    # ============================================================

## 6. Kiểm tra trước khi huấn luyện lâu

Trước khi chạy nhiều epoch, hãy chạy **một mini-batch**.

Nếu code đúng:

- loss phải hữu hạn;
- logits không chứa NaN;
- sau `loss.backward()`, ít nhất một parameter phải có gradient khác 0.

Bước kiểm tra nhỏ này rẻ hơn rất nhiều so với phát hiện lỗi sau 20 phút huấn luyện.

In [ ]:
sanity_model = SmallCNN(
    use_batchnorm=True,
    dropout_p=0.1,
).to(device)

criterion = nn.CrossEntropyLoss()

xb, yb = next(
    iter(train_loader)
)

xb = xb.to(device)
yb = yb.to(device)

sanity_logits = sanity_model(
    xb
)

sanity_loss = criterion(
    sanity_logits,
    yb,
)

print(
    "Initial loss:",
    sanity_loss.item(),
)

sanity_loss.backward()

grad_norms = [
    p.grad.norm().item()
    for p in sanity_model.parameters()
    if p.grad is not None
]

assert torch.isfinite(
    sanity_loss
)
assert max(grad_norms) > 0

print(
    "Max gradient norm:",
    max(grad_norms),
)
print("✓ Kiểm tra mini-batch hợp lệ")

## 7. Mô hình cơ sở

Bắt đầu từ một cấu hình cố định trước khi tìm kiếm.

Ví dụ:

- BatchNorm: bật;
- Dropout: 0.1;
- optimizer: Adam;
- learning rate: $10^{-3}$;
- 5 epoch.

Với 10 lớp, mức đoán ngẫu nhiên là khoảng 10%. Mô hình CNN đúng phải nhanh chóng vượt
xa mức này.

Không cần đạt một con số giống hệt nhau giữa mọi máy; hãy tập trung vào xu hướng loss
giảm và độ chính xác kiểm định tăng.

In [ ]:
def fit_cnn(
    config,
    epochs=5,
    seed=42,
):
    set_seed(seed)

    train_loader_local, val_loader_local, _ = (
        get_cifar10_loaders(
            batch_size=config[
                "batch_size"
            ],
            seed=seed,
        )
    )

    model = SmallCNN(
        use_batchnorm=config[
            "use_batchnorm"
        ],
        dropout_p=config[
            "dropout_p"
        ],
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config[
            "weight_decay"
        ],
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
    }

    for epoch in range(epochs):
        tr_loss, tr_acc = (
            train_one_epoch(
                model,
                train_loader_local,
                criterion,
                optimizer,
                device,
            )
        )

        va_loss, va_acc = evaluate(
            model,
            val_loader_local,
            criterion,
            device,
        )

        history[
            "train_loss"
        ].append(tr_loss)

        history[
            "train_acc"
        ].append(tr_acc)

        history[
            "val_loss"
        ].append(va_loss)

        history[
            "val_acc"
        ].append(va_acc)

        print(
            f"Epoch {epoch + 1:02d} | "
            f"train loss {tr_loss:.4f} | "
            f"train acc {tr_acc:.3f} | "
            f"val acc {va_acc:.3f}"
        )

    return model, history

### Thực hành 4 — Huấn luyện mô hình cơ sở

In [ ]:
baseline_config = {
    "use_batchnorm": True,
    "dropout_p": 0.1,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "batch_size": 128,
}

# ============================================================
# TODO
#
# Huấn luyện mô hình cơ sở trong 5 epoch.
# Vẽ:
# - train/validation loss;
# - train/validation accuracy.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 2 — Đọc các đường cong huấn luyện

Dựa trên mô hình cơ sở của bạn:

1. Loss có giảm đều không?
2. Chênh lệch độ chính xác huấn luyện–kiểm định có lớn không?
3. Nếu validation loss bắt đầu tăng trong khi train loss vẫn giảm, hiện tượng này gợi ý gì?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 8. Tìm kiếm siêu tham số có giới hạn

Assignment này còn phần RNN, vì vậy không nên dành toàn bộ thời gian cho CNN.

**Ngân sách:** tối đa **6 cấu hình**, gồm cả mô hình cơ sở.

Hãy thay đổi có chủ đích ít nhất ba yếu tố:

- learning rate;
- BatchNorm bật/tắt;
- Dropout;
- weight decay;
- batch size.

Giữ seed cố định để so sánh công bằng hơn.

In [ ]:
results = []

# ============================================================
# TODO
#
# Chạy tối đa 6 cấu hình, tính cả baseline.
#
# Với mỗi cấu hình, lưu ít nhất:
# - use_batchnorm
# - dropout_p
# - lr
# - weight_decay
# - batch_size
# - final train_acc
# - final val_acc
# ============================================================

results_df = (
    pd.DataFrame(results)
    .sort_values(
        "val_acc",
        ascending=False,
    )
    if len(results) > 0
    else pd.DataFrame()
)

results_df

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 3 — Chuẩn hóa theo lô và Dropout trong CNN

Dựa trên các lần chạy của bạn:

1. Tắt BatchNorm làm thay đổi tốc độ tối ưu như thế nào?
2. Dropout quá lớn có thể làm độ chính xác huấn luyện giảm vì sao?
3. Cấu hình có độ chính xác huấn luyện cao nhất có nhất thiết là cấu hình nên chọn không?

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## 9. Lựa chọn mô hình cuối cùng

Chọn cấu hình dựa trên **độ chính xác kiểm định**, không nhìn tập kiểm tra (test set) trong lúc tìm kiếm.

Sau đó:

1. huấn luyện lại mô hình cuối;
2. đánh giá tập kiểm tra (test set) **một lần**;
3. lưu `best_cnn.pt`.

Q5 sẽ dùng `forward_features()` của CNN để tạo vector đặc trưng ảnh cho RNN.

Ta lưu bằng TorchScript để Q5 không cần khai báo lại `SmallCNN`.

In [ ]:
def save_cnn_for_q5(
    model,
    path="best_cnn.pt",
):
    model_cpu = model.to(
        "cpu"
    ).eval()

    scripted = torch.jit.script(
        model_cpu
    )

    torch.jit.save(
        scripted,
        path,
    )


# ============================================================
# TODO
#
# 1. Chọn best_config từ validation results.
# 2. Huấn luyện/retrain final_model.
# 3. Đánh giá test set đúng một lần.
# 4. Lưu bằng:
#       save_cnn_for_q5(final_model)
# 5. Giữ lại test accuracy trong output notebook.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

## 10. Quan sát đặc trưng đã học

Một cách trực quan đơn giản là xem activation của các kênh đầu hoặc xem ảnh nào được mô
hình phân loại đúng/sai.

Không cần một công cụ giải thích phức tạp. Mục tiêu là liên hệ:

$$
\text{ảnh}
\rightarrow
\text{bản đồ đặc trưng (feature maps)}
\rightarrow
\text{vector đặc trưng}
\rightarrow
\text{logits}.
$$

### Thực hành 5 — Phân tích một số kết quả dự đoán

In [ ]:
# ============================================================
# TODO
#
# Hiển thị ít nhất 8 ảnh từ validation/test cùng:
# - nhãn thật;
# - nhãn dự đoán;
# - xác suất lớn nhất.
#
# Chọn cả ví dụ đúng và sai.
# ============================================================

raise NotImplementedError("Hoàn thiện phần thực hành này")

### Câu hỏi 4 — Lỗi của CNN

Quan sát các mẫu sai:

- Những cặp lớp nào dễ nhầm?
- Lỗi có liên quan đến hình dạng, nền ảnh hoặc độ phân giải thấp của CIFAR-10 không?

Nêu ít nhất hai quan sát cụ thể từ output của bạn.

### ✍️ Câu trả lời của bạn

<!-- Viết câu trả lời bên dưới. Không xóa ô này. -->

## Tóm tắt Q3

Bạn đã:

- xây một CNN bằng PyTorch;
- sử dụng BatchNorm và Dropout;
- triển khai vòng lặp train/evaluation;
- điều chỉnh siêu tham số bằng tập kiểm định (validation set);
- lưu một CNN cuối để dùng như bộ trích xuất đặc trưng ở Q5.